# Qian — LoRA versus AdaLoRA target-rank sweep
Four equal-time arms on the official augreg2 ViT-B/16: **LoRA r24**, and **AdaLoRA initial rank 24 → target average rank 8 / 16 / 20**. The primary metric is final holdout accuracy under the same <5% allocated backbone parameter cap and training-time budget.

Use **seed 42 only**, the same 1,000-class split (40 fit + 5 holdout/class), fitting-only prototypes, 10,000-way cosine head and extras as the LoRA/DoRA experiment. Train at **448px**, 75% random crop + 25% whole-image/padding; **batch64 × accumulation4 = effective256**. Evaluation is crop0.9 with no flip or zoom.

Default: **12 training minutes per arm, 48 total**, plus memory probe, prototypes, validation and saving. An A100 40GB is recommended. This is a subset screen, not a full-data submission run. Existing LoRA/DoRA outputs stay separate.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/qq_adalora_repo"
BRANCH = "qq_lora_ablation"  # Commit and push the implementation before running in Colab.
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
EXPERIMENT_NAME = "qq_adalora_448_c1000_seed42_v1"
CONFIG = "configs/qq_adalora_ablation.yaml"
MINUTES_PER_ARM = 12
NUM_CLASSES_SUBSET = 1000
NUM_WORKERS = 8
DOWNLOAD_DATA = True  # Existing extracted images are reused.
# All arms use seed42. Use a new experiment name after changing code/settings/
# hardware or after an interrupted arm. Finished arms with identical specs are reused.


## 1. Connect to GitHub
Same hidden-token connection as your training launcher. Checkout/pull `qq_lora_ablation`; this does not merge it with main.


In [ ]:
import sys
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


## 2. Install packages and check the GPU


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-adalora.txt"], check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime; A100 is recommended."
print("GPU:", torch.cuda.get_device_name())
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if 'A100' not in torch.cuda.get_device_name():
    print("For comparable timing, run all arms on the same A100. Do not mix hardware between arms.")


## 3. Mount Drive
Use the same `CS604_RUNS` shared-folder shortcut as before. Dataset images stay on temporary Colab storage; results and checkpoint files go to Drive.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
base_out = Path(DRIVE_OUT)
if not base_out.is_dir():
    raise FileNotFoundError(f"{DRIVE_OUT} not found. Add the shared CS604_RUNS folder as a shortcut in My Drive.")
EXPERIMENT_DIR = base_out / EXPERIMENT_NAME
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
print("Experiment:", EXPERIMENT_DIR)


## 4. Download or reuse extracted images
The full download/extraction can be slow and needs room for archives plus images. Check disk space first. Set `DOWNLOAD_DATA=False` to use an existing extract. Reuse is based on a small path check; it does not certify every file.


In [ ]:
# Stream subprocess output into the notebook, and stop on failure.
def run_live(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
        process.wait()
        raise


import shutil
root = Path(DATA_DIR)
root.mkdir(parents=True, exist_ok=True)
print(f"Free Colab disk: {shutil.disk_usage(root).free / 2**30:.1f} GiB")
def extracted_training_available():
    listing = root / 'train.txt'
    if not listing.is_file(): return False
    paths = []
    with listing.open() as handle:
        for line in handle:
            if line.strip(): paths.append(line.split()[0])
            if len(paths) == 20: break
    return bool(paths) and all((root / path).is_file() for path in paths)
if DOWNLOAD_DATA and not extracted_training_available():
    run_live([sys.executable, '-u', 'scripts/download_data.py', '--dest', DATA_DIR])
else:
    print('Using existing dataset:', root)
assert (root / 'train.txt').is_file(), 'Set DATA_DIR to the extracted training dataset.'


## 5. Review settings and the parameter preflight
The preflight constructs untrained CPU models, counts actual allocated parameters (including AdaLoRA singular values), and downloads no pretrained weights. All arms must fit strictly below 5%, **including their initial allocation**. AdaLoRA starts at rank24 in all 48 target matrices, so its three arms initially allocate the same parameters.

The implementation uses PEFT 0.21.2 SVDLinear and RankAllocator. AdaLoRA's cubic rank-budget schedule uses elapsed training time: **10% initial phase, reduction through 80%, final20% with a fixed mask**. Rank allocation runs after optimizer updates before clearing gradients. Orthogonality regularization uses PEFT's coefficient **0.5** and is added once per optimizer update. The LR schedule remains the shared 5% time warmup + cosine.

A target is an average over matrices, not a uniform rank per layer or an exact parameter percentage. Rank costs differ by matrix shape. Actual ranks and active parameter counts are logged; masked tensors retain initial rank24 in memory.


In [ ]:
from src.utils import load_config
cfg = load_config(CONFIG)
print("Backbone:", cfg.model_name)
print("Resolution / padding:", cfg.img_size, cfg.full_image_probability)
print("LoRA rank / AdaLoRA initial rank:", cfg.lora_rank, cfg.adalora_init_rank)
print("AdaLoRA target averages:", cfg.adalora_target_ranks)
print("Batch / accumulation:", cfg.batch_size, cfg.grad_accum_steps)
print("Seed / split / prototypes:", cfg.seed, cfg.split_seed, cfg.prototype_seed)
print("Adapter / head LR:", cfg.lr, cfg.head_lr)
print("Minutes per arm / total:", MINUTES_PER_ARM, MINUTES_PER_ARM * 4)
run_live([sys.executable, '-u', '-m', 'src.qq_adalora_ablation', '--config', CONFIG,
          '--preflight-only', '--minutes', str(MINUTES_PER_ARM),
          '--classes', str(NUM_CLASSES_SUBSET), '--workers', str(NUM_WORKERS)])


## 6. Run all four arms
Before any timed run, the runner checks AdaLoRA forward/backward, orthogonal regularization, AdamW state, importance statistics and rank masking using synthetic inputs. All AdaLoRA targets allocate the same initial tensors, so target8 is used for this memory probe. If it fails, no timed arm starts. Real-data allocation can still differ.

Each timed arm starts from fresh pretrained weights and the same fitting-only prototypes. Image loading, augmentation, allocation updates and regularization count toward its 12-minute budget; prototype extraction, validation and saving do not. Results are saved after each completed arm. Completed arms are reused only when code, settings, dependency versions, hardware, split and checkpoints match. Partial runs are not silently resumed.


In [ ]:
command = [sys.executable, '-u', '-m', 'src.qq_adalora_ablation',
           '--config', CONFIG, '--data_root', DATA_DIR, '--output_dir', str(EXPERIMENT_DIR),
           '--minutes', str(MINUTES_PER_ARM), '--classes', str(NUM_CLASSES_SUBSET),
           '--workers', str(NUM_WORKERS)]
run_live(command)


## 7. Compare against LoRA r24
Rank candidates by **final accuracy after equal training time**. Best-epoch accuracy is context only. Compare throughput, epochs and peak VRAM alongside accuracy. `final_active_pct` includes extras and excludes the classifier; `adaptation_pct` counts the full allocated trainable tensors throughout training.

Check `allocation_frozen` for each AdaLoRA arm. If false, the target/final training phase was not reached; that run cannot establish performance at the intended target budget. Close differences on this one fixed holdout remain inconclusive. There is no automatic full-data run or seed43 repeat.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
results = pd.read_csv(EXPERIMENT_DIR / 'results.csv')
display(results[['arm', 'final_val_acc', 'best_val_acc', 'final_val_loss', 'training_minutes',
                 'epochs', 'images_per_second', 'peak_vram_gib', 'adaptation_pct',
                 'final_average_rank', 'final_active_pct', 'allocation_frozen']])
comparison = json.loads((EXPERIMENT_DIR / 'comparison.json').read_text())
for row in comparison['comparisons']:
    print(f"{row['arm']} minus LoRA r24: {row['delta_pp']:+.2f} percentage points; "
          f"throughput {row['speed_ratio']:.3f}x; final average rank {row['final_average_rank']:.2f}; "
          f"allocation fixed: {row['allocation_frozen']}")
if comparison['best_adalora']:
    best = comparison['best_adalora']
    print(f"Best AdaLoRA target: {best['target_rank']}, final accuracy {best['final_val_acc']:.2%}")
print(comparison['note'])
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, field, title in zip(axes, ['final_val_acc', 'images_per_second', 'peak_vram_gib'],
                           ['Final holdout accuracy (%)', 'Training images / second', 'Peak memory (GiB)']):
    values = results[field] * 100 if field == 'final_val_acc' else results[field]
    ax.bar(results.arm, values)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / 'comparison.png', dpi=150)
plt.show()


## 8. Inspect learned ranks and training curves
Per-run `adalora_rank_log.csv` records the scheduled budget, actual active ranks and regularization; `adalora_final.json` records the final distribution. Masking may retain fewer components than the target when importance scores tie. Full adapter tensors remain allocated; lower active rank does not guarantee lower peak VRAM.

The cell below exports per-layer ranks across the three target budgets. A full-data run would require fresh pretrained weights and all-data prototypes plus a separate 144-minute budget. These subset scores do not predict a full-data winner with certainty.


In [ ]:
rank_rows = []
for row in results.itertuples(index=False):
    run_live([sys.executable, 'scripts/plot_curves.py', '--run_dir', row.run_dir,
              '--out_dir', str(EXPERIMENT_DIR / 'seed42' / row.arm / 'plots')])
    if row.method == 'adalora':
        final = json.loads((Path(row.run_dir) / 'adalora_final.json').read_text())
        rank_rows.extend({'arm': row.arm, 'layer': layer, 'active_rank': rank}
                         for layer, rank in final['ranks'].items())
rank_table = pd.DataFrame(rank_rows)
rank_table.to_csv(EXPERIMENT_DIR / 'rank_allocations.csv', index=False)
display(rank_table.pivot(index='layer', columns='arm', values='active_rank'))
print('Saved results:', EXPERIMENT_DIR)
